[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/D-Barradas/Accelerated-Data-Science-with-RAPIDS/blob/main/part4/4-03_Zero_Copy_Optuna_Lightning_RAPIDS.ipynb)

# Part 4-3: Zero-Copy Deep Tabular Learning with RAPIDS, Optuna, and PyTorch Lightning

**Series:** Accelerated Data Science with RAPIDS — Part 4: Hyperparameter Optimization

## What you will learn

A common hidden bottleneck in GPU-accelerated ML pipelines is the *CPU-GPU memory round-trip*: preprocessing data on the GPU (e.g. with cuDF), then silently copying it back to host RAM to build a PyTorch tensor, then copying it back to the GPU again to train. For large tabular datasets this round-trip can dominate total pipeline time.

This notebook eliminates that bottleneck end-to-end:

1. Load and clean a large real-world tabular dataset (airline on-time performance, 2003) with **cuDF**, entirely on the GPU.
2. Convert the cleaned cuDF DataFrames **directly** into PyTorch tensors using **DLPack** — a zero-copy, framework-agnostic GPU memory standard.
3. Define a configurable MLP classifier as a **PyTorch Lightning** `LightningModule`, which removes training-loop boilerplate so we can focus on the model and the HPO search.
4. Use **Optuna** with the `PyTorchLightningPruningCallback` to search the hyperparameter space efficiently, aborting unpromising trials early instead of letting every trial run to completion.

## 1. Setup & Environment Verification

Run the cell below once per Colab session. We pin `cudf-cu12` from NVIDIA's package index (RAPIDS wheels are not published on the default PyPI index) and install PyTorch Lightning, Optuna, the Optuna PyTorch Lightning integration, and `gdown` (used later to fetch the dataset from Google Drive).

> **Tip:** If any import fails right after installing, use **Runtime -> Restart runtime** (not "restart and run all") and re-run from the top — this clears any partially-initialized CUDA context left by the installer.

In [ ]:
# Install RAPIDS cuDF (cu12 wheels) from NVIDIA's package index, plus PyTorch Lightning,
#!pip -q install --extra-index-url=https://pypi.nvidia.com cudf-cu12 cuml-cu12 cupy-cuda12x || true

# Core ML stack
!pip -q install pytorch-lightning optuna optuna-integration[pytorch_lightning] 

In [ ]:
# Verify GPU availability and confirm the versions of every library this notebook depends on.
import subprocess

import cudf
import optuna
import pytorch_lightning as pl
import torch

print("=" * 60)
print("GPU (nvidia-smi)")
print("=" * 60)
try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    )
    print(smi.stdout)
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    print(f"nvidia-smi unavailable: {exc}. Make sure the Colab runtime type is set to GPU.")

print("=" * 60)
print("Library versions")
print("=" * 60)
print(f"torch             : {torch.__version__}")
print(f"torch.cuda avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"torch CUDA device : {torch.cuda.get_device_name(0)}")
print(f"cudf              : {cudf.__version__}")
print(f"pytorch_lightning : {pl.__version__}")
print(f"optuna            : {optuna.__version__}")

assert torch.cuda.is_available(), (
    "No CUDA device found. In Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4/L4)."
)

## 2. Data Ingestion & Google Drive Mount

We store the 2003 airline on-time performance dataset in Google Drive so it survives across Colab sessions, and download it once with `gdown` if it isn't already there. The cell checks for the file before downloading, so re-running this notebook against the same Drive won't re-fetch multi-hundred-MB data every time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import gdown

dir_path = '/content/drive/MyDrive/Accel_DS_RAPIDS'
file_path = 'part4/data/airline-data-full-2003.orc'
absolute_path_to_file = os.path.join(dir_path, file_path)

print(f"Checking existence of directory: {os.path.dirname(absolute_path_to_file)}")
os.makedirs(os.path.dirname(absolute_path_to_file), exist_ok=True)

print(f"\nChecking existence of file: {absolute_path_to_file}")
if os.path.exists(absolute_path_to_file):
    print(f"The file '{file_path}' exists.")
else:
    print(f"The file '{file_path}' not found.")
    print(f"Downloading...")
    url = "https://drive.google.com/uc?id=1EecyIkVpGMWBm3AHGfCQij4YN92Cjddf"
    gdown.download(url, absolute_path_to_file, quiet=False)

## 3. Accelerated Data Pipeline & Zero-Copy Transfer

We load the ORC file with `cudf.read_orc`, which decodes it directly onto the GPU. All cleaning, encoding, and scaling below runs as cuDF GPU kernels — the data never touches host (CPU) memory during preprocessing. We define a binary target: whether a flight arrives more than 15 minutes late.

In [ ]:
# Additional imports used by the data pipeline, model, and training loop below.
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

In [ ]:
# Load the airline on-time performance data directly onto the GPU.
gdf = cudf.read_orc(absolute_path_to_file)
print(f"Rows: {len(gdf):,}  |  Columns: {len(gdf.columns)}")
gdf.head()

In [ ]:
# Drop cancelled/diverted flights (no meaningful arrival delay) and rows missing key fields.
NUMERIC_FEATURES = ["Month", "DayofMonth", "DayOfWeek", "CRSDepTime", "CRSArrTime", "Distance"]
CATEGORICAL_FEATURES = ["UniqueCarrier", "Origin", "Dest"]
DELAY_THRESHOLD_MIN = 15

gdf = gdf[gdf["Cancelled"] == 0]
if "Diverted" in gdf.columns:
    gdf = gdf[gdf["Diverted"] == 0]
gdf = gdf.dropna(subset=["ArrDelay"] + NUMERIC_FEATURES + CATEGORICAL_FEATURES).reset_index(drop=True)

# Binary classification target: did the flight arrive more than 15 minutes late?
gdf["is_delayed"] = (gdf["ArrDelay"] > DELAY_THRESHOLD_MIN).astype("int32")

print(f"Rows after cleaning: {len(gdf):,}")
print(f"Positive (delayed) rate: {gdf['is_delayed'].mean():.3f}")

In [ ]:
# Encode categorical flight attributes as integer codes, then scale every feature to
# zero mean / unit variance -- both are GPU-vectorized cuDF operations over millions of rows.
encoded = gdf[NUMERIC_FEATURES].astype("float32")

for col in CATEGORICAL_FEATURES:
    encoded[col] = gdf[col].astype("category").cat.codes.astype("float32")

feature_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
feature_means = encoded[feature_cols].mean()
feature_stds = encoded[feature_cols].std() + 1e-6
encoded[feature_cols] = (encoded[feature_cols] - feature_means) / feature_stds

target = gdf["is_delayed"].astype("float32")

print(f"Feature matrix shape: {encoded.shape}")
encoded.head()

### Why DLPack?

Normally, moving data from a GPU DataFrame into a PyTorch tensor means: copy GPU -> host RAM (as pandas/NumPy), then copy host RAM -> GPU again for `torch`. For a dataset with millions of rows, that round-trip dominates preprocessing time and doubles peak memory usage.

**DLPack** is an open, framework-agnostic tensor memory standard. `cudf` can export a DataFrame's underlying GPU buffer as a DLPack capsule with `to_dlpack()`, and PyTorch can import that *exact same GPU memory* with `torch.utils.dlpack.from_dlpack()` — no host copy, no new GPU allocation, just a reinterpretation of the existing buffer as a `torch.Tensor`.

In [ ]:
# Zero-copy cuDF -> PyTorch conversion using the DLPack standard.
# `to_dlpack()` exports the GPU buffer as a DLPack capsule; `from_dlpack()` reinterprets that
# same GPU memory as a torch.Tensor -- no host round-trip, no extra GPU allocation.
from torch.utils.dlpack import from_dlpack


def cudf_to_tensor(data) -> torch.Tensor:
    """Zero-copy conversion of a cuDF DataFrame/Series to a torch.Tensor via DLPack."""
    dlpack_capsule = data.to_dlpack()
    return from_dlpack(dlpack_capsule)


# .contiguous() normalizes the tensor's memory layout if the DLPack view is column-major;
# this is a cheap on-GPU reshape, not a host round-trip.
X = cudf_to_tensor(encoded[feature_cols]).contiguous()
y = cudf_to_tensor(target).contiguous().unsqueeze(1)

print(f"X: {tuple(X.shape)} {X.dtype} on {X.device}")
print(f"y: {tuple(y.shape)} {y.dtype} on {y.device}")

In [ ]:
# Wrap the zero-copy tensors in a train/validation split and PyTorch DataLoaders.
# num_workers must stay 0 (the default): the tensors already live on the GPU, and CUDA
# tensors cannot be shared across the extra worker processes a DataLoader would spawn.
dataset = TensorDataset(X, y)
n_val = int(0.2 * len(dataset))
n_train = len(dataset) - n_val
train_dataset, val_dataset = random_split(
    dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42),
)

BATCH_SIZE = 512
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 4. PyTorch Lightning Model Definition

**Why PyTorch Lightning?** Writing a training loop by hand means re-implementing the same boilerplate every time: device placement, `optimizer.zero_grad()`/`backward()`/`step()`, `model.train()`/`model.eval()` switching, metric aggregation across batches. PyTorch Lightning's `LightningModule` factors all of that out into a small number of well-defined methods (`training_step`, `validation_step`, `configure_optimizers`), so the same model definition scales from a quick experiment to multi-GPU training without touching the training loop — which matters a lot once we start running dozens of Optuna trials.

The architecture (depth, width, dropout) and the optimizer's learning rate are all constructor arguments, so Optuna can treat them as tunable hyperparameters.

In [ ]:
class TabularMLP(pl.LightningModule):
    """Configurable MLP classifier for tabular flight-delay prediction.

    Parameters
    ----------
    input_dim : number of input features.
    hidden_dims : sizes of each hidden layer, e.g. [128, 64].
    dropout : dropout probability applied after every hidden layer.
    lr : Adam learning rate.
    """

    def __init__(self, input_dim: int, hidden_dims: list, dropout: float, lr: float):
        super().__init__()
        self.save_hyperparameters()

        layers = []
        in_features = input_dim
        for out_features in hidden_dims:
            layers += [nn.Linear(in_features, out_features), nn.ReLU(), nn.Dropout(dropout)]
            in_features = out_features
        layers.append(nn.Linear(in_features, 1))

        self.network = nn.Sequential(*layers)
        self.criterion = nn.BCEWithLogitsLoss()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.criterion(self(x), y)
        self.log("train_loss", loss, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = (torch.sigmoid(logits) > 0.5).float()
        acc = (preds == y).float().mean()
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_acc", acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

In [ ]:
# Quick sanity check: one forward pass with random weights before handing off to Optuna.
_sanity_model = TabularMLP(input_dim=X.shape[1], hidden_dims=[64, 32], dropout=0.1, lr=1e-3)
_sanity_out = _sanity_model(X[:8].cpu())
assert _sanity_out.shape == (8, 1), f"Unexpected output shape: {_sanity_out.shape}"
print("Model forward pass OK:", tuple(_sanity_out.shape))

## 5. Optuna Integration & Pruning

**Why Optuna with pruning?** A naive grid/random search trains every configuration to completion, even ones that are clearly bad after a couple of epochs. Optuna's pruning API lets a trial report its intermediate validation metric after every epoch; a **pruner** (here, `MedianPruner`) compares that value against other trials at the same step and aborts the run immediately if it's falling behind. The `PyTorchLightningPruningCallback` wires this decision directly into the PyTorch Lightning training loop, so pruning happens with no manual bookkeeping.

In [ ]:
# optuna-integration renamed/relocated this callback across Optuna versions; support both.
try:
    from optuna_integration import PyTorchLightningPruningCallback
except ImportError:
    from optuna.integration import PyTorchLightningPruningCallback

In [ ]:
def objective(trial: optuna.Trial) -> float:
    """Train one MLP configuration and return its validation loss for Optuna to minimize."""
    # Search space: architecture depth/width, dropout, and learning rate.
    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128, 256])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)

    model = TabularMLP(
        input_dim=X.shape[1],
        hidden_dims=[hidden_size] * n_layers,
        dropout=dropout,
        lr=lr,
    )

    # Reports `val_loss` back to Optuna after every validation epoch, and requests this
    # trial be stopped early (via `check_pruned` below) if it is underperforming.
    pruning_callback = PyTorchLightningPruningCallback(trial, monitor="val_loss")

    trainer = pl.Trainer(
        max_epochs=15,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        enable_checkpointing=False,
        callbacks=[pruning_callback],
    )

    try:
        trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
    except torch.cuda.OutOfMemoryError as exc:
        # Free the allocation and let Optuna treat this configuration as a failed trial
        # instead of crashing the whole study.
        torch.cuda.empty_cache()
        raise optuna.TrialPruned(f"Trial pruned after CUDA OOM: {exc}")

    # Re-raises optuna.TrialPruned if the callback flagged this trial mid-training.
    pruning_callback.check_pruned()

    return trainer.callback_metrics["val_loss"].item()

## 6. Execution & Analysis

We now launch the Optuna study. `MedianPruner` stops a trial once its intermediate score is worse than the median of previous trials at the same epoch — cheap, unpromising configurations are killed fast, leaving more wall-clock time for promising ones.

In [ ]:
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
)
study.optimize(objective, n_trials=20, show_progress_bar=True)

In [ ]:
# Summarize the best trial found by the search.
pruned_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
complete_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

print(f"Finished trials : {len(study.trials)}")
print(f"Pruned trials   : {len(pruned_trials)}")
print(f"Completed trials: {len(complete_trials)}")
print(f"\nBest validation loss: {study.best_value:.4f}")
print("Best hyperparameters:")
for key, val in study.best_params.items():
    print(f"  {key:>12}: {val}")

In [ ]:
# Visualize the HPO search: how the score improved over trials, and which hyperparameters
# mattered most for predicting flight delays.
fig_history = optuna.visualization.plot_optimization_history(study)
fig_history.show()

fig_importance = optuna.visualization.plot_param_importances(study)
fig_importance.show()

In [ ]:
# Clean resource teardown: release GPU memory held by the driver process.
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("CUDA cache cleared.")

## Summary

- **cuDF** cleaned, encoded, and scaled a large tabular flight dataset entirely on the GPU, with no CPU preprocessing bottleneck.
- **DLPack** (`to_dlpack()` / `from_dlpack()`) moved the resulting features straight into PyTorch tensors with zero host-memory copies.
- **PyTorch Lightning** removed training-loop boilerplate, letting the model definition stay focused on architecture while scaling cleanly across dozens of Optuna trials.
- **Optuna's pruning** (`MedianPruner` + `PyTorchLightningPruningCallback`) killed unpromising trials early, concentrating compute on the hyperparameter regions most likely to improve validation loss.